In [ ]:
!pip install pandas numpy requests beautifulsoup4 lxml hmmlearn matplotlib scikit-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 15.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 16.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 14.6 MB/s  0:00:00 eta 0:00:01
  Created wheel for hmmlearn: filename=hmmlearn-0.3.3-cp314-cp314-macosx_10_15_universal2.whl size=230208 sha256=ec4b19486bcb805ad578cdcd2447e42a6d20db6ee4dae84c9c577f1810118471
  Stored in directory: /Users/dustinlee/Library/Caches/pip/wheels/bf/ba/1d/7dca6f4dcaba999e3914ea8c1e9e9287923bb95c420863df25
Successfully built hmmlearn
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [pandas]learn]

In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from hmmlearn import hmm
import matplotlib.pyplot as plt
import time

Start with 118th congress, 1st session

In [ ]:
BASE = "https://www.congress.gov"
session_url = "https://www.congress.gov/votes/house/118th-congress/1st-session"

html = requests.get(session_url).text
soup = BeautifulSoup(html, "html.parser")

vote_links = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/votes/house/118-1/" in href:
        full_url = BASE + href if href.startswith("/") else href
        vote_links.append(full_url)

vote_links = sorted(set(vote_links))

print("Number of vote pages:", len(vote_links))
vote_links[:5]

scrape the votes

In [ ]:
def scrape_vote_page(url):
    vote_id = url.rstrip("/").split("/")[-1]

    try:
        tables = pd.read_html(url)
    except Exception as e:
        print("Failed:", url, e)
        return None

    for table in tables:
        columns = set(table.columns.astype(str))
        if {"Representative", "Party", "State", "Vote"}.issubset(columns):
            table = table.copy()
            table["vote_id"] = vote_id
            table["url"] = url
            return table

    return None


dfs = []

for i, url in enumerate(vote_links[:50]):  # start small first
    df = scrape_vote_page(url)
    if df is not None:
        dfs.append(df)

    time.sleep(0.25)

votes_df = pd.concat(dfs, ignore_index=True)
votes_df.head()

encode the votes

In [ ]:
vote_map = {
    "Yea": 1,
    "Aye": 1,
    "Nay": 0,
    "No": 0,
    "Present": 2,
    "Not Voting": 3
}

votes_df["vote_encoded"] = votes_df["Vote"].map(vote_map)
votes_df = votes_df.dropna(subset=["vote_encoded"])
votes_df["vote_encoded"] = votes_df["vote_encoded"].astype(int)

votes_df.head()

build the matrix

In [ ]:
vote_matrix = votes_df.pivot_table(
    index="vote_id",
    columns="Representative",
    values="vote_encoded",
    aggfunc="first"
)

vote_matrix = vote_matrix.fillna(3)
vote_matrix = vote_matrix.astype(int)

vote_matrix.head()

train the HMM for one rep

In [ ]:
rep = vote_matrix.columns[0]

X = vote_matrix[[rep]].values

model = hmm.CategoricalHMM(
    n_components=3,
    n_iter=100,
    random_state=42
)

model.fit(X)

hidden_states = model.predict(X)

result = pd.DataFrame({
    "vote_id": vote_matrix.index,
    "representative": rep,
    "vote_encoded": vote_matrix[rep].values,
    "hidden_state": hidden_states
})

result.head(20)

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(result["hidden_state"], marker="o")
plt.title(f"Hidden Voting States Over Time: {rep}")
plt.xlabel("Vote Index")
plt.ylabel("Hidden State")
plt.show()